# Equity Daily-Feature SJM Teacher

Equity-only copy of the teacher-labeling logic from `sjm_ensembles.ipynb`, with teacher features computed from daily SPX levels and sampled to monthly decision dates.

- Daily features use `1m=21d`, `3m=63d`, `6m=126d`, `12m=252d`.
- EWMA half-lives use `h1=21d`, `h2=42d`, `h4=84d`.
- First SJM fit starts after `144` monthly observations.
- Fits both `SJM_L2` on causal expanding z-scores and `SJM_L1_MEDOIDS` on causal expanding robust z-scores.
- Builds the same three equity label families: risk, downside return-risk, and drawdown return-risk.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans

DATA_XLSX = Path.home() / "Downloads" / "data_ale (1).xlsx"
SHEET = "Sheet1"
EPS, TDY = 1e-12, 252
MIN_TRAIN_OBS, REFIT_EVERY = 144, 1
LAMBDA_GRID = [0, 0.25, 0.5, 1, 2, 4, 8, 16, 32, 64]
MAX_CD_ITER, N_INIT, RANDOM_SEED = 50, 10, 123

def title(x):
    print("\n" + "=" * 160)
    print(x)
    print("=" * 160)

# -----------------------------------------------------------------------------
# 1. Daily equity features, month-end sampled
# -----------------------------------------------------------------------------
def read_bbg(path=DATA_XLSX, sheet=SHEET):
    raw = pd.read_excel(path, sheet_name=sheet, header=None)
    h = raw.index[raw.iloc[:, 0].astype(str).str.strip().eq("Date")][0]
    x = pd.read_excel(path, sheet_name=sheet, header=h)[["Date", "PX_LAST"]]
    x = x.rename(columns={"Date": "date", "PX_LAST": "eq_level"})
    x["date"] = pd.to_datetime(x["date"])
    x["eq_level"] = pd.to_numeric(x["eq_level"], errors="coerce")
    return x.dropna().sort_values("date").drop_duplicates("date", keep="last").reset_index(drop=True)

def daily_to_monthly_equity_features():
    d = read_bbg(); P = d["eq_level"]; r = P.pct_change()
    out = pd.DataFrame({"date": d["date"], "eq_level": P, "eq_ret_1d": r})
    out["eq_dd_exp"] = P / P.expanding().max() - 1
    for label, w in {"1m": 21, "3m": 63, "6m": 126, "12m": 252}.items():
        mu = TDY * r.rolling(w, min_periods=w).mean()
        vol = np.sqrt(TDY) * r.rolling(w, min_periods=w).std()
        down = np.sqrt(TDY) * np.sqrt((np.minimum(r, 0) ** 2).rolling(w, min_periods=w).mean())
        peak = P.rolling(w, min_periods=w).max(); trough = P.rolling(w, min_periods=w).min()
        out[f"eq_ret_{label}"] = P / P.shift(w) - 1
        out[f"eq_vol_{label}"] = vol
        out[f"eq_downside_{label}"] = down
        out[f"eq_sharpe_{label}"] = mu / (vol + EPS)
        out[f"eq_sortino_{label}"] = mu / (down + EPS)
        out[f"eq_peak_{label}"] = peak
        out[f"eq_dd_{label}"] = P / peak - 1
        out[f"eq_recovery_{label}"] = P / trough - 1
        out[f"eq_tail10_{label}"] = r.rolling(w, min_periods=w).quantile(.10)
        out[f"eq_ddspeed_{label}"] = out["eq_dd_exp"] - out["eq_dd_exp"].shift(w)
    out["eq_worst_3m"] = out["eq_ret_3m"].rolling(63, min_periods=63).min()
    out["eq_dvol_3m"] = out["eq_vol_3m"] - out["eq_vol_3m"].shift(63)
    out["eq_volratio_1_3"] = out["eq_vol_1m"] / (out["eq_vol_3m"] + EPS)
    for h, hl in {"h1": 21, "h2": 42, "h4": 84}.items():
        mean = TDY * r.ewm(halflife=hl, adjust=False, min_periods=hl).mean()
        std = np.sqrt(TDY) * r.ewm(halflife=hl, adjust=False, min_periods=hl).std()
        downside = np.sqrt(TDY) * np.sqrt((np.minimum(r, 0) ** 2).ewm(halflife=hl, adjust=False, min_periods=hl).mean())
        out[f"eq_mean_{h}"] = mean
        out[f"eq_std_{h}"] = std
        out[f"eq_sortino_{h}"] = mean / (downside + EPS)
        out[f"eq_sharpe_{h}"] = mean / (std + EPS)
        out[f"eq_logdd_{h}"] = np.log(downside + EPS)
        out[f"eq_dd_{h}"] = out["eq_dd_exp"].ewm(halflife=hl, adjust=False, min_periods=hl).mean()
        out[f"eq_recovery_{h}"] = out["eq_recovery_3m"].ewm(halflife=hl, adjust=False, min_periods=hl).mean()
    out["eq_ddspeed_h2"] = out["eq_dd_h2"] - out["eq_dd_h2"].shift(21)
    out["month_end"] = out["date"] + pd.offsets.MonthEnd(0)
    m = out.groupby("month_end", as_index=False).last().drop(columns="date").rename(columns={"month_end": "date"})
    m["equity_excess"] = m["eq_level"].pct_change()
    m["equity_dd_realized"] = m["eq_level"] / m["eq_level"].expanding().max() - 1
    return d, m.replace([np.inf, -np.inf], np.nan)

EWM_RETURN_RISK_DOWNSIDE_EQ = ["eq_mean_h1", "eq_mean_h2", "eq_mean_h4", "eq_sortino_h1", "eq_sortino_h2", "eq_sortino_h4", "eq_logdd_h1", "eq_logdd_h2", "eq_logdd_h4"]
EWM_RETURN_RISK_DRAWDOWN_EQ = [*EWM_RETURN_RISK_DOWNSIDE_EQ, "eq_dd_h1", "eq_dd_h4", "eq_ddspeed_h2", "eq_recovery_h4"]
SHORT_TAIL_VOL_EQ = ["eq_vol_1m", "eq_vol_3m", "eq_dvol_3m", "eq_volratio_1_3", "eq_downside_1m", "eq_downside_3m", "eq_logdd_h1", "eq_worst_3m", "eq_tail10_6m"]
FEATURE_GROUPS_RAW = {"SHORT_TAIL_VOL": SHORT_TAIL_VOL_EQ, "EWM_RETURN_RISK_DOWNSIDE": EWM_RETURN_RISK_DOWNSIDE_EQ, "EWM_RETURN_RISK_DRAWDOWN": EWM_RETURN_RISK_DRAWDOWN_EQ}

# -----------------------------------------------------------------------------
# 2. Same causal standardization and SJM machinery as the main notebook
# -----------------------------------------------------------------------------
def standardize(df, cols):
    out = df.copy()
    for c in cols:
        mu = out[c].expanding(60).mean().shift(1); sd = out[c].expanding(60).std().shift(1)
        med = out[c].expanding(60).median().shift(1); q75 = out[c].expanding(60).quantile(.75).shift(1); q25 = out[c].expanding(60).quantile(.25).shift(1)
        out[c + "_z"] = (out[c] - mu) / sd.replace(0, np.nan)
        out[c + "_rz"] = (out[c] - med) / (q75 - q25).replace(0, np.nan)
    return out

def dp(loss, lam):
    V = loss.copy(); B = np.zeros_like(loss, dtype=int); n = len(loss)
    for t in range(1, n):
        for s in [0, 1]:
            stay, switch = V[t - 1, s], V[t - 1, 1 - s] + lam
            B[t, s] = s if stay <= switch else 1 - s; V[t, s] += min(stay, switch)
    path = np.zeros(n, dtype=int); path[-1] = int(np.argmin(V[-1]))
    for t in range(n - 2, -1, -1): path[t] = B[t + 1, path[t + 1]]
    return path

def fit_sjm_l2_window(X, lam):
    y = KMeans(n_clusters=2, n_init=10, random_state=RANDOM_SEED).fit_predict(X)
    for _ in range(MAX_CD_ITER):
        C = np.vstack([X[y == k].mean(0) if np.any(y == k) else X.mean(0) for k in [0, 1]])
        yn = dp(np.column_stack([((X - C[k]) ** 2).sum(1) for k in [0, 1]]), lam)
        if np.array_equal(y, yn): break
        y = yn
    return y

def fit_sjm_l1_medoids_window(X, lam):
    rng = np.random.default_rng(RANDOM_SEED); y = KMeans(n_clusters=2, n_init=10, random_state=RANDOM_SEED).fit_predict(X)
    for _ in range(MAX_CD_ITER):
        M = []
        for k in [0, 1]:
            Xk = X[y == k]
            M.append(X[int(rng.integers(len(X)))] if len(Xk) == 0 else Xk[np.argmin(cdist(Xk, Xk, "cityblock").sum(1))])
        M = np.vstack(M); yn = dp(np.column_stack([np.abs(X - M[k]).sum(1) for k in [0, 1]]), lam)
        if np.array_equal(y, yn): break
        y = yn
    return y

def n_switches(s):
    s = pd.Series(s).dropna().astype(int); return int((s.diff().dropna() != 0).sum()) if len(s) else 0
def spell(s):
    s = pd.Series(s).dropna(); return len(s) / (1 + n_switches(s)) if len(s) else np.nan
def ann_vol(x):
    x = pd.Series(x).dropna(); return np.sqrt(12) * x.std() if len(x) > 1 else np.nan
def ann_sharpe(x):
    x = pd.Series(x).dropna(); return 12 * x.mean() / (ann_vol(x) + EPS) if len(x) > 2 else np.nan


# Plot helpers must be defined before the SJM grid because individual paths are plotted immediately after each lambda fit.
STATE_COLORS = {1: "#b7e4bd", 0: "#f5b5b5"}

def fmt_pct(x):
    return "NA" if not np.isfinite(x) else f"{100*x:0.1f}%"

def fmt_num(x):
    return "NA" if not np.isfinite(x) else f"{x:0.2f}"

def fmt_m(x):
    return "NA" if not np.isfinite(x) else f"{x:0.1f}m"

def date_edges(dates):
    dates = pd.to_datetime(pd.Series(dates)).sort_values().reset_index(drop=True)
    if len(dates) == 0:
        return pd.Series(dtype="datetime64[ns]"), pd.Series(dtype="datetime64[ns]")
    if len(dates) == 1:
        return pd.Series([dates.iloc[0] - pd.offsets.MonthBegin(1)]), pd.Series([dates.iloc[0] + pd.offsets.MonthEnd(1)])
    mids = dates.iloc[:-1] + (dates.iloc[1:].to_numpy() - dates.iloc[:-1].to_numpy()) / 2
    left = pd.Series([dates.iloc[0] - (mids.iloc[0] - dates.iloc[0]), *list(mids)])
    right = pd.Series([*list(mids), dates.iloc[-1] + (dates.iloc[-1] - mids.iloc[-1])])
    return left, right

def add_state_background_runs(ax, df, state_col):
    x = df[["date", state_col]].dropna().copy()
    if x.empty:
        return
    x["date"] = pd.to_datetime(x["date"])
    x[state_col] = x[state_col].astype(int)
    x = x.sort_values("date").reset_index(drop=True)
    left, right = date_edges(x["date"])
    x["left"] = left
    x["right"] = right
    x["run_id"] = (x[state_col] != x[state_col].shift()).cumsum()
    ymin, ymax = ax.get_ylim()
    for _, g in x.groupby("run_id", sort=True):
        ax.axvspan(
            g["left"].iloc[0],
            g["right"].iloc[-1],
            facecolor=STATE_COLORS[int(g[state_col].iloc[0])],
            alpha=0.60,
            edgecolor="none",
            linewidth=0,
            antialiased=False,
            zorder=0,
        )
    ax.set_ylim(ymin, ymax)

def run_causal_sjm(df, group, cols, model_family, feature_source, lam):
    X = df[cols].to_numpy(float); r = df["equity_excess"].to_numpy(float); dd = df["equity_dd_realized"].to_numpy(float)
    valid = np.isfinite(X).all(1) & np.isfinite(r); state = np.full(len(df), np.nan); raw_state = np.full(len(df), np.nan); seen = []
    fit = fit_sjm_l2_window if model_family == "SJM_L2" else fit_sjm_l1_medoids_window
    for t in range(len(df)):
        if valid[t]: seen.append(t)
        if t % REFIT_EVERY or not valid[t] or len(seen) < MIN_TRAIN_OBS: continue
        idx = np.asarray(seen); labels = fit(X[idx], lam); raw_last = int(labels[-1])
        metric = [ann_vol(r[idx][labels == k]) if group == "SHORT_TAIL_VOL" else ann_sharpe(r[idx][labels == k]) for k in [0, 1]]
        raw_state1 = int(metric[1] > metric[0]) if np.all(np.isfinite(metric)) else 1
        raw_state[t] = raw_last; state[t] = int(raw_last == raw_state1)
    return pd.DataFrame({"date": df["date"], "asset": "equity", "model_family": model_family, "feature_group": group, "feature_source": feature_source, "lambda": lam, "state_raw": raw_state, "state_id": state, "q_state1": state, "q_good": state, "asset_excess": r, "asset_drawdown": dd})

# Original-style immediate plot for each individual fitted teacher path.
def plot_teacher_path_now(panel, group, family, source, lam, level_frame):
    good_state = 0 if group == "SHORT_TAIL_VOL" else 1
    state_col = "plot_good_state"
    x = panel[["date", "state_id", "asset_excess", "asset_drawdown"]].copy()
    x[state_col] = x["state_id"] if good_state == 1 else 1.0 - x["state_id"]
    x = x.merge(level_frame[["date", "eq_level"]], on="date", how="left").dropna(subset=[state_col, "eq_level", "asset_excess"]).sort_values("date")
    if x.empty:
        print(f"No plottable labels for {family} | {group} | lambda={lam:g}")
        return

    good, bad = x[state_col].eq(1), x[state_col].eq(0)
    line1 = f"Obs: {len(x)}   Good avg: {100*good.mean():0.1f}%   Switches: {n_switches(x[state_col])}   Spell: {spell(x[state_col]):0.1f}m"
    line2 = f"Ann. mean good/bad: {100*12*x.loc[good,'asset_excess'].mean():0.1f}% / {100*12*x.loc[bad,'asset_excess'].mean():0.1f}%   Sharpe good/bad: {ann_sharpe(x.loc[good,'asset_excess']):0.2f} / {ann_sharpe(x.loc[bad,'asset_excess']):0.2f}"
    line3 = f"Avg full-path DD good/bad: {100*x.loc[good,'asset_drawdown'].mean():0.1f}% / {100*x.loc[bad,'asset_drawdown'].mean():0.1f}%"

    fig = plt.figure(figsize=(16, 4.0))
    gs = GridSpec(2, 1, height_ratios=[3.2, .95], hspace=.08, figure=fig)
    ax = fig.add_subplot(gs[0, 0]); ax_stats = fig.add_subplot(gs[1, 0])
    ax.plot(x["date"], x["eq_level"], color="black", linewidth=1.25, zorder=5)
    add_state_background_runs(ax, x, state_col)
    ax.set_title(f"{family} | {source} | {group} | lambda={lam:g}", loc="left", fontsize=12, fontweight="bold")
    ax.set_ylabel("Equity index"); ax.grid(True, alpha=.25)
    ax.legend(handles=[Patch(facecolor=STATE_COLORS[1], alpha=.60, label="Good"), Patch(facecolor=STATE_COLORS[0], alpha=.60, label="Bad")], loc="upper left", frameon=True)
    ax_stats.axis("off"); ax_stats.text(.01, .72, line1 + "\n" + line2 + "\n" + line3, ha="left", va="top", fontsize=10, family="monospace")
    plt.tight_layout(); plt.show()

# -----------------------------------------------------------------------------
# 3. Fit all equity SJM teachers and majority-vote targets
# -----------------------------------------------------------------------------
title("DAILY EQUITY FEATURE CONSTRUCTION")
daily_spx, monthly_raw = daily_to_monthly_equity_features()
all_raw = sorted(set(c for cols in FEATURE_GROUPS_RAW.values() for c in cols))
work = standardize(monthly_raw, all_raw)
print("Source:", DATA_XLSX)
print("Daily SPX panel:", daily_spx.shape, daily_spx.date.min().date(), "to", daily_spx.date.max().date())
print("Month-end teacher panel:", monthly_raw.shape, monthly_raw.date.min().date(), "to", monthly_raw.date.max().date())
print("Raw teacher features:", len(all_raw))
print("Feature groups:", {k: len(v) for k, v in FEATURE_GROUPS_RAW.items()})
display(monthly_raw[["date", "equity_excess", *all_raw]].tail())

title("CAUSAL EQUITY SJM GRID")
print("MIN_TRAIN_OBS:", MIN_TRAIN_OBS)
print("REFIT_EVERY:", REFIT_EVERY)
print("Lambda grid:", LAMBDA_GRID)
print("Model families: SJM_L2 on STD_Z, SJM_L1_MEDOIDS on ROBUST_RZ")

teacher_parts = []
for group, raw_cols in FEATURE_GROUPS_RAW.items():
    for family, suffix, source in [("SJM_L2", "_z", "STD_Z"), ("SJM_L1_MEDOIDS", "_rz", "ROBUST_RZ")]:
        feature_cols = [c + suffix for c in raw_cols]
        print(f"{family} | {group} | {source} | features={len(feature_cols)}")
        for lam in LAMBDA_GRID:
            print(f"  lambda={lam:g}")
            one_teacher = run_causal_sjm(work, group, feature_cols, family, source, lam)
            teacher_parts.append(one_teacher)
            plot_teacher_path_now(one_teacher, group, family, source, lam, work)
teacher_panel_all = pd.concat(teacher_parts, ignore_index=True)

teacher_results = (teacher_panel_all.dropna(subset=["state_id"])
    .groupby(["asset", "model_family", "feature_group", "feature_source", "lambda"])
    .agg(n_obs_signal=("state_id", "size"), state1_share=("state_id", "mean"), n_switches=("state_id", n_switches), mean_spell_length=("state_id", spell), teacher_sharpe=("asset_excess", ann_sharpe), first_date=("date", "min"), last_date=("date", "max"))
    .reset_index())

selected_specs = teacher_results.query("n_obs_signal > 0 and state1_share > 0 and state1_share < 1").copy()
selected_specs["spec_id"] = (selected_specs["model_family"] + "__" + selected_specs["feature_group"] + "__" + selected_specs["feature_source"] + "__lam" + selected_specs["lambda"].astype(str))

TARGET_SPECS = pd.DataFrame([
    {"feature_group": "SHORT_TAIL_VOL", "state_col": "eq_risk_state", "prob_col": "eq_risk_prob", "good_state": 0, "label": "Risk state"},
    {"feature_group": "EWM_RETURN_RISK_DOWNSIDE", "state_col": "eq_downside_state", "prob_col": "eq_downside_prob", "good_state": 1, "label": "Return-risk / downside state"},
    {"feature_group": "EWM_RETURN_RISK_DRAWDOWN", "state_col": "eq_drawdown_state", "prob_col": "eq_drawdown_prob", "good_state": 1, "label": "Drawdown-aware state"},
])

keys = ["model_family", "feature_group", "feature_source", "lambda"]
tp_valid = teacher_panel_all.merge(selected_specs[keys + ["spec_id"]], on=keys, how="inner")
target_wide = pd.DataFrame({"date": work["date"]})
family_vote_wide = pd.DataFrame({"date": work["date"]})

for _, spec in TARGET_SPECS.iterrows():
    group, state_col, prob_col, good_state = spec["feature_group"], spec["state_col"], spec["prob_col"], int(spec["good_state"])
    sub = tp_valid[tp_valid["feature_group"].eq(group)]
    raw_vote = sub.groupby("date")["state_id"].mean()
    raw_majority = (raw_vote >= 0.5).astype(float)
    out = pd.DataFrame({"date": raw_vote.index, "state1_prob": raw_vote.values, "state1_majority": raw_majority.values})
    out[prob_col] = out["state1_prob"] if good_state == 1 else 1.0 - out["state1_prob"]
    out[state_col] = out["state1_majority"] if good_state == 1 else 1.0 - out["state1_majority"]
    target_wide = target_wide.merge(out[["date", state_col, prob_col]], on="date", how="left")

    for family in ["SJM_L2", "SJM_L1_MEDOIDS"]:
        fs = sub[sub["model_family"].eq(family)]
        fv = fs.groupby("date")["state_id"].mean()
        fm = (fv >= 0.5).astype(float)
        fcol = f"{state_col}__{family}"
        fpcol = f"{prob_col}__{family}"
        fdf = pd.DataFrame({"date": fv.index, fpcol: fv.values, fcol: fm.values})
        if good_state == 0:
            fdf[fpcol] = 1.0 - fdf[fpcol]
            fdf[fcol] = 1.0 - fdf[fcol]
        family_vote_wide = family_vote_wide.merge(fdf, on="date", how="left")

for c in ["eq_risk_state", "eq_downside_state", "eq_drawdown_state"]:
    target_wide[c + "_h1"] = target_wide[c].shift(-1)

target_wide = target_wide.merge(work[["date", "eq_level", "equity_excess", "equity_dd_realized"]], on="date", how="left")
family_vote_wide = family_vote_wide.merge(work[["date", "eq_level", "equity_excess", "equity_dd_realized"]], on="date", how="left")
target_wide["equity_level"] = 100 * (1 + target_wide["equity_excess"].fillna(0)).cumprod()
family_vote_wide["equity_level"] = 100 * (1 + family_vote_wide["equity_excess"].fillna(0)).cumprod()

FEATURE_ORDER = ["SHORT_TAIL_VOL", "EWM_RETURN_RISK_DOWNSIDE", "EWM_RETURN_RISK_DRAWDOWN"]
LABEL_LOOKUP = TARGET_SPECS.set_index("feature_group")["label"].to_dict()
STATE_COL_LOOKUP = TARGET_SPECS.set_index("feature_group")["state_col"].to_dict()

def state_metrics(df, state_col, target_name):
    x = df.dropna(subset=[state_col, "equity_excess"]).copy()
    rows = []
    for s in [0, 1]:
        r = x.loc[x[state_col].eq(s), "equity_excess"]
        rows.append({
            "target": target_name, "state": s, "n": len(r), "share": len(r) / len(x) if len(x) else np.nan,
            "ann_return": 12 * r.mean(), "ann_vol": np.sqrt(12) * r.std(), "sharpe": ann_sharpe(r),
            "avg_drawdown": x.loc[x[state_col].eq(s), "equity_dd_realized"].mean(),
        })
    return rows

diagnostic_rows = []
for state_col, name in [("eq_risk_state", "risk_current"), ("eq_downside_state", "downside_current"), ("eq_drawdown_state", "drawdown_current"), ("eq_risk_state_h1", "risk_h1"), ("eq_downside_state_h1", "downside_h1"), ("eq_drawdown_state_h1", "drawdown_h1")]:
    diagnostic_rows.extend(state_metrics(target_wide, state_col, name))
majority_vote_metrics = pd.DataFrame(diagnostic_rows)

def stats_text(df, state_col):
    x = df.dropna(subset=[state_col, "equity_excess", "equity_dd_realized"]).copy(); x[state_col] = x[state_col].astype(int)
    good, bad = x[state_col].eq(1), x[state_col].eq(0)
    return (
        f"Obs: {len(x)}   Good avg: {fmt_pct(good.mean())}   Switches: {n_switches(x[state_col])}   Spell: {fmt_m(spell(x[state_col]))}\n"
        f"Ann. mean good/bad: {fmt_pct(12*x.loc[good,'equity_excess'].mean())} / {fmt_pct(12*x.loc[bad,'equity_excess'].mean())}   "
        f"Sharpe good/bad: {fmt_num(ann_sharpe(x.loc[good,'equity_excess']))} / {fmt_num(ann_sharpe(x.loc[bad,'equity_excess']))}\n"
        f"Avg full-path DD good/bad: {fmt_pct(x.loc[good,'equity_dd_realized'].mean())} / {fmt_pct(x.loc[bad,'equity_dd_realized'].mean())}"
    )

def plot_three_panel(df, plot_cols, title_prefix):
    fig = plt.figure(figsize=(16, 10.5)); gs = GridSpec(6, 1, height_ratios=[3.2, .95, 3.2, .95, 3.2, .95], hspace=.10, figure=fig)
    axes = []
    for i, (feature_group, state_col) in enumerate(plot_cols):
        ax = fig.add_subplot(gs[2*i, 0]); ax_stats = fig.add_subplot(gs[2*i+1, 0]); axes.append(ax)
        x = df[["date", state_col, "equity_level", "equity_excess", "equity_dd_realized"]].dropna(subset=[state_col, "equity_level"]).sort_values("date")
        ax.plot(x["date"], x["equity_level"], color="black", linewidth=1.3, zorder=5)
        add_state_background_runs(ax, x, state_col)
        ax.set_title(f"{title_prefix} | Equity | {LABEL_LOOKUP.get(feature_group, feature_group)}", loc="left", fontsize=12, fontweight="bold")
        ax.set_ylabel("Equity index"); ax.grid(True, alpha=.25)
        ax_stats.axis("off"); ax_stats.text(.01, .70, stats_text(x, state_col), ha="left", va="top", fontsize=10, family="monospace")
    axes[0].legend(handles=[Patch(facecolor=STATE_COLORS[1], alpha=.60, label="Good/state 1"), Patch(facecolor=STATE_COLORS[0], alpha=.60, label="Bad/state 0")], loc="upper left", frameon=True)
    plt.tight_layout(); plt.show()

title("EQUITY DAILY-FEATURE SJM TEACHER RESULTS")
display(teacher_results.sort_values(["feature_group", "model_family", "lambda"]))

title("VALID SJM TEACHER PATHS")
valid_summary = (selected_specs.groupby(["feature_group", "model_family", "feature_source"])
    .agg(n_specs=("lambda", "size"), median_switches=("n_switches", "median"), median_spell=("mean_spell_length", "median"), median_state1_share=("state1_share", "median"), median_teacher_sharpe=("teacher_sharpe", "median"))
    .reset_index())
display(valid_summary)

title("EQUITY MAJORITY-VOTE TARGET METRICS")
display(majority_vote_metrics)
display(target_wide.dropna().tail())

title("L1/L2 FAMILY-AGGREGATED LABEL PLOTS")
plot_three_panel(
    family_vote_wide,
    [(g, f"{STATE_COL_LOOKUP[g]}__SJM_L2") for g in FEATURE_ORDER],
    "SJM_L2 aggregate across valid lambdas",
)
plot_three_panel(
    family_vote_wide,
    [(g, f"{STATE_COL_LOOKUP[g]}__SJM_L1_MEDOIDS") for g in FEATURE_ORDER],
    "SJM_L1_MEDOIDS aggregate across valid lambdas",
)

title("ALL-VALID-SJM MAJORITY-VOTE LABEL PLOT")
plot_three_panel(
    target_wide,
    [(g, STATE_COL_LOOKUP[g]) for g in FEATURE_ORDER],
    "All-valid SJM majority vote",
)
